In [ ]:

# ==========================
# 1. Bibliotecas
# ==========================
import truststore
truststore.inject_into_ssl()

import msal
import requests
import pandas as pd

# ==========================
# 2. Configuração
# ==========================
CLIENT_ID = "8c7f39b0-ffaf-4cbc-8b21-307306bc06a6"
TENANT_ID = "8d0c8ea3-0505-44ff-b7a9-6520ea97aacb"

# ==========================
# 3. Autenticação
# ==========================
app = msal.PublicClientApplication(
    client_id=CLIENT_ID,
    authority=f"https://login.microsoftonline.com/{TENANT_ID}"
)

flow = app.initiate_device_flow(scopes=["Mail.Read"])

if "user_code" not in flow:
    raise RuntimeError(flow)

print(flow["message"])

token = app.acquire_token_by_device_flow(flow)

if "access_token" not in token:
    raise RuntimeError(token.get("error_description", token))

print("Autenticação concluída.")

# ==========================
# 4. Ler emails
# ==========================
url = "https://graph.microsoft.com/v1.0/me/mailFolders/inbox/messages"

headers = {
    "Authorization": f"Bearer {token['access_token']}",
    "Prefer": 'outlook.body-content-type="text"'
}

params = {
    "$top": 100,
    "$select": "id,receivedDateTime,from,subject,body",
    "$orderby": "receivedDateTime desc"
}

response = requests.get(
    url,
    headers=headers,
    params=params,
    timeout=30
)

response.raise_for_status()
emails = response.json()["value"]

# ==========================
# 5. DataFrame
# ==========================
df = pd.DataFrame([
    {
        "ID": email.get("id"),
        "Data": email.get("receivedDateTime"),
        "Remetente": (
            (email.get("from") or {}).get("emailAddress") or {}
        ).get("address"),
        "Assunto": email.get("subject"),
        "Mensagem": (email.get("body") or {}).get("content")
    }
    for email in emails
])

df


To sign in, use a web browser to open the page https://login.microsoft.com/device and enter the code GWGTD7GGG to authenticate.
